# 10 - Ensemble y Mejoras Finales

## Objetivo: Romper el 60% Top-1

Estrategias:
1. **Ensemble LSTM + GRU**: Combinar predicciones de ambos modelos
2. **WLASL100 filtrado**: Usar mas datos de las mismas 30 clases
3. **Label Smoothing**: Regularizacion para mejorar generalizacion

## Estado actual
- LSTM (notebook 09): Val 37.5%, Test 41.2%
- GRU suave (notebook 09b): Val 53.6%, Test 38.2%

In [6]:
# ==========================================
# 1. IMPORTS
# ==========================================
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Tuple
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [7]:
# ==========================================
# 2. RUTAS
# ==========================================
BASE = Path("/home/fransando/Dev/tfg-sign-language-recognition")
EXPERIMENTS = BASE / "05_experiments"
DATASETS = BASE / "06_datasets/wlasl/landmarks"

# Modelos existentes
LSTM_MODEL_PATH = EXPERIMENTS / "lstm_landmarks/best_model.pt"
GRU_MODEL_PATH = EXPERIMENTS / "lstm_landmarks_wlasl30_gru_suave/best_model.pt"

# Datasets
WLASL30_ROOT = DATASETS / "wlasl30"
WLASL100_ROOT = DATASETS / "wlasl100"

# Output
OUTPUT_DIR = EXPERIMENTS / "ensemble_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"LSTM model: {LSTM_MODEL_PATH.exists()}")
print(f"GRU model: {GRU_MODEL_PATH.exists()}")

LSTM model: True
GRU model: True


---
# PARTE 1: ENSEMBLE DE MODELOS EXISTENTES
---

In [8]:
# ==========================================
# 3. DEFINIR ARQUITECTURA (igual que antes)
# ==========================================

class PoseClassifier(nn.Module):
    def __init__(self, config: Dict):
        super().__init__()
        
        input_size = config['input_size']
        hidden_size = config['hidden_size']
        num_layers = config['num_layers']
        num_classes = config['num_classes']
        dropout = config['dropout']
        bidirectional = config['bidirectional']
        model_type = config.get('model_type', 'LSTM')
        
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        RNN_CLASS = nn.LSTM if model_type == 'LSTM' else nn.GRU
        self.rnn = RNN_CLASS(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        self.model_type = model_type
        
        rnn_output_size = hidden_size * self.num_directions
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(rnn_output_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        
        if self.model_type == 'LSTM':
            rnn_out, (h_n, c_n) = self.rnn(x)
        else:
            rnn_out, h_n = self.rnn(x)
        
        if self.bidirectional:
            h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_final = h_n[-1]
        
        return self.classifier(h_final)

print("PoseClassifier definido.")

PoseClassifier definido.


In [9]:
# ==========================================
# 4. CARGAR MODELOS
# ==========================================

# Cargar checkpoints
lstm_ckpt = torch.load(LSTM_MODEL_PATH, map_location=device)
gru_ckpt = torch.load(GRU_MODEL_PATH, map_location=device)

print("LSTM config:", lstm_ckpt['config'])
print("\nGRU config:", gru_ckpt['config'])

LSTM config: {'max_frames': 64, 'num_landmarks': 21, 'num_coords': 3, 'input_size': 63, 'aug_rotation_range': 15, 'aug_scale_range': (0.9, 1.1), 'aug_translation_range': 0.1, 'aug_noise_std': 0.01, 'hidden_size': 256, 'num_layers': 2, 'dropout': 0.4, 'bidirectional': True, 'batch_size': 32, 'epochs': 100, 'lr': 0.001, 'weight_decay': 0.0001, 'num_classes': 30}

GRU config: {'max_frames': 64, 'input_size': 63, 'aug_rotation_range': 10, 'aug_scale_range': (0.9, 1.1), 'aug_translation_range': 0.1, 'aug_noise_std': 0.01, 'aug_prob': 0.5, 'aug_landmark_mask_prob': 0.0, 'aug_frame_mask_prob': 0.0, 'model_type': 'GRU', 'hidden_size': 256, 'num_layers': 2, 'dropout': 0.4, 'bidirectional': True, 'batch_size': 32, 'epochs': 150, 'lr': 0.001, 'weight_decay': 0.0005, 'patience': 30, 'num_classes': 30}


In [10]:
# ==========================================
# 5. CREAR MODELOS Y CARGAR PESOS
# ==========================================

# LSTM
lstm_config = lstm_ckpt['config'].copy()
lstm_config['model_type'] = 'LSTM'
lstm_model = PoseClassifier(lstm_config).to(device)

# Renombrar claves del state_dict: lstm.* -> rnn.*
lstm_state_dict = lstm_ckpt['model_state_dict']
lstm_state_dict_fixed = {}
for key, value in lstm_state_dict.items():
	new_key = key.replace('lstm.', 'rnn.') if key.startswith('lstm.') else key
	lstm_state_dict_fixed[new_key] = value

lstm_model.load_state_dict(lstm_state_dict_fixed)
lstm_model.eval()
print(f"LSTM cargado (Val Top-1: {lstm_ckpt.get('val_top1', 'N/A'):.1f}%)")

# GRU
gru_config = gru_ckpt['config'].copy()
gru_model = PoseClassifier(gru_config).to(device)
gru_model.load_state_dict(gru_ckpt['model_state_dict'])
gru_model.eval()
print(f"GRU cargado (Val Top-1: {gru_ckpt.get('val_top1', 'N/A'):.1f}%)")

LSTM cargado (Val Top-1: 37.5%)
GRU cargado (Val Top-1: 53.6%)


In [11]:
# ==========================================
# 6. DATASET Y LOADER
# ==========================================

class PoseDataset(Dataset):
    def __init__(self, root_dir: Path, split: str, gloss_to_idx: Dict,
                 max_frames: int = 64, augmentation=None):
        self.root_dir = root_dir
        self.max_frames = max_frames
        self.augmentation = augmentation
        self.gloss_to_idx = gloss_to_idx
        
        self.samples = []
        split_dir = root_dir / split
        
        for gloss_dir in split_dir.iterdir():
            if not gloss_dir.is_dir():
                continue
            gloss = gloss_dir.name
            if gloss not in gloss_to_idx:
                continue
            label = gloss_to_idx[gloss]
            
            for npy_file in gloss_dir.glob('*.npy'):
                self.samples.append((npy_file, label, gloss))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        npy_path, label, gloss = self.samples[idx]
        landmarks = np.load(npy_path)
        landmarks = self._pad_or_truncate(landmarks)
        
        if self.augmentation:
            landmarks = self.augmentation(landmarks)
        
        landmarks = landmarks.reshape(self.max_frames, -1)
        return torch.tensor(landmarks, dtype=torch.float32), label
    
    def _pad_or_truncate(self, landmarks):
        T = landmarks.shape[0]
        if T > self.max_frames:
            indices = np.linspace(0, T-1, self.max_frames, dtype=int)
            landmarks = landmarks[indices]
        elif T < self.max_frames:
            pad_size = self.max_frames - T
            padding = np.zeros((pad_size, 21, 3))
            landmarks = np.concatenate([landmarks, padding], axis=0)
        return landmarks


# Cargar metadata
with open(WLASL30_ROOT / 'metadata.json') as f:
    meta = json.load(f)

gloss_to_idx = meta['gloss_to_idx']
idx_to_gloss = {int(k): v for k, v in meta['idx_to_gloss'].items()}

# Crear loaders
val_dataset = PoseDataset(WLASL30_ROOT, 'val', gloss_to_idx)
test_dataset = PoseDataset(WLASL30_ROOT, 'test', gloss_to_idx)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

Val: 56 samples
Test: 34 samples


In [12]:
# ==========================================
# 7. FUNCION DE EVALUACION CON ENSEMBLE
# ==========================================

@torch.no_grad()
def evaluate_ensemble(models: List[nn.Module], loader: DataLoader, 
                      weights: List[float] = None, top_k=(1, 5)):
    """
    Evalua un ensemble de modelos.
    
    Args:
        models: Lista de modelos
        loader: DataLoader
        weights: Pesos para cada modelo (default: iguales)
        top_k: Tuple de k para calcular top-k accuracy
    """
    if weights is None:
        weights = [1.0 / len(models)] * len(models)
    
    for m in models:
        m.eval()
    
    total = 0
    top_k_correct = {k: 0 for k in top_k}
    all_preds, all_labels = [], []
    
    for batch, labels in loader:
        batch, labels = batch.to(device), labels.to(device)
        
        # Sumar logits ponderados de todos los modelos
        ensemble_logits = None
        for model, w in zip(models, weights):
            logits = model(batch)
            if ensemble_logits is None:
                ensemble_logits = w * logits
            else:
                ensemble_logits += w * logits
        
        total += labels.size(0)
        
        _, pred_topk = ensemble_logits.topk(max(top_k), 1, True, True)
        for k in top_k:
            correct_k = pred_topk[:, :k].eq(labels.view(-1, 1).expand_as(pred_topk[:, :k]))
            top_k_correct[k] += correct_k.any(dim=1).sum().item()
        
        all_preds.extend(ensemble_logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    metrics = {}
    for k in top_k:
        metrics[f'top{k}_acc'] = 100 * top_k_correct[k] / total
    
    return metrics, np.array(all_preds), np.array(all_labels)


# Evaluar modelos individuales primero
print("=" * 50)
print("EVALUACION INDIVIDUAL")
print("=" * 50)

lstm_val, _, _ = evaluate_ensemble([lstm_model], val_loader)
lstm_test, _, _ = evaluate_ensemble([lstm_model], test_loader)
print(f"LSTM:  Val Top-1: {lstm_val['top1_acc']:.1f}%  Test Top-1: {lstm_test['top1_acc']:.1f}%")

gru_val, _, _ = evaluate_ensemble([gru_model], val_loader)
gru_test, _, _ = evaluate_ensemble([gru_model], test_loader)
print(f"GRU:   Val Top-1: {gru_val['top1_acc']:.1f}%  Test Top-1: {gru_test['top1_acc']:.1f}%")

EVALUACION INDIVIDUAL
LSTM:  Val Top-1: 37.5%  Test Top-1: 41.2%
GRU:   Val Top-1: 53.6%  Test Top-1: 38.2%


In [13]:
# ==========================================
# 8. ENSEMBLE LSTM + GRU
# ==========================================

print("\n" + "=" * 50)
print("ENSEMBLE LSTM + GRU")
print("=" * 50)

# Probar diferentes pesos
weight_configs = [
    ([0.5, 0.5], "50/50"),
    ([0.4, 0.6], "40/60 (mas GRU)"),
    ([0.6, 0.4], "60/40 (mas LSTM)"),
    ([0.3, 0.7], "30/70 (mucho mas GRU)"),
]

best_config = None
best_val = 0

for weights, name in weight_configs:
    val_metrics, _, _ = evaluate_ensemble([lstm_model, gru_model], val_loader, weights)
    test_metrics, _, _ = evaluate_ensemble([lstm_model, gru_model], test_loader, weights)
    
    print(f"{name}: Val {val_metrics['top1_acc']:.1f}% / Test {test_metrics['top1_acc']:.1f}% (Top-5: {test_metrics['top5_acc']:.1f}%)")
    
    if val_metrics['top1_acc'] > best_val:
        best_val = val_metrics['top1_acc']
        best_config = (weights, name, val_metrics, test_metrics)

print(f"\nMejor config: {best_config[1]}")
print(f"  Val Top-1: {best_config[2]['top1_acc']:.1f}%")
print(f"  Test Top-1: {best_config[3]['top1_acc']:.1f}%")
print(f"  Test Top-5: {best_config[3]['top5_acc']:.1f}%")


ENSEMBLE LSTM + GRU
50/50: Val 46.4% / Test 52.9% (Top-5: 79.4%)
40/60 (mas GRU): Val 46.4% / Test 52.9% (Top-5: 79.4%)
60/40 (mas LSTM): Val 46.4% / Test 50.0% (Top-5: 73.5%)
30/70 (mucho mas GRU): Val 50.0% / Test 52.9% (Top-5: 79.4%)

Mejor config: 30/70 (mucho mas GRU)
  Val Top-1: 50.0%
  Test Top-1: 52.9%
  Test Top-5: 79.4%


---
# PARTE 2: WLASL100 FILTRADO A 30 CLASES
---

In [14]:
# ==========================================
# 9. CREAR DATASET WLASL100 -> 30 CLASES
# ==========================================

# Las 30 clases de WLASL30
TARGET_GLOSSES = set(gloss_to_idx.keys())
print(f"Clases objetivo: {len(TARGET_GLOSSES)}")

# Crear nuevo dataset filtrado
FILTERED_ROOT = DATASETS / "wlasl100_filtered30"

def create_filtered_dataset():
    """Crea un dataset con las 30 clases de WLASL30 usando datos de WLASL100."""
    
    stats = {'train': 0, 'val': 0, 'test': 0}
    
    for split in ['train', 'val', 'test']:
        src_dir = WLASL100_ROOT / split
        dst_dir = FILTERED_ROOT / split
        
        for gloss_dir in src_dir.iterdir():
            if not gloss_dir.is_dir():
                continue
            
            gloss = gloss_dir.name
            if gloss not in TARGET_GLOSSES:
                continue
            
            # Copiar archivos
            dst_gloss_dir = dst_dir / gloss
            dst_gloss_dir.mkdir(parents=True, exist_ok=True)
            
            for npy_file in gloss_dir.glob('*.npy'):
                shutil.copy2(npy_file, dst_gloss_dir / npy_file.name)
                stats[split] += 1
    
    return stats

# Crear si no existe
if not FILTERED_ROOT.exists():
    print("Creando dataset filtrado...")
    stats = create_filtered_dataset()
    print(f"Creado: {stats}")
else:
    # Contar
    stats = {}
    for split in ['train', 'val', 'test']:
        count = len(list((FILTERED_ROOT / split).glob('**/*.npy')))
        stats[split] = count
    print(f"Dataset filtrado ya existe: {stats}")

Clases objetivo: 30
Creando dataset filtrado...
Creado: {'train': 242, 'val': 56, 'test': 34}


In [15]:
# ==========================================
# 10. COMPARAR TAMAÑOS
# ==========================================

print("\n" + "=" * 50)
print("COMPARACION DE DATASETS")
print("=" * 50)

def count_samples(root):
    counts = {}
    for split in ['train', 'val', 'test']:
        counts[split] = len(list((root / split).glob('**/*.npy')))
    return counts

wlasl30_counts = count_samples(WLASL30_ROOT)
filtered_counts = count_samples(FILTERED_ROOT)

print(f"\n{'Dataset':<25} {'Train':<10} {'Val':<10} {'Test':<10} {'Total':<10}")
print("-" * 65)
print(f"{'WLASL30 original':<25} {wlasl30_counts['train']:<10} {wlasl30_counts['val']:<10} {wlasl30_counts['test']:<10} {sum(wlasl30_counts.values()):<10}")
print(f"{'WLASL100 filtrado':<25} {filtered_counts['train']:<10} {filtered_counts['val']:<10} {filtered_counts['test']:<10} {sum(filtered_counts.values()):<10}")

improvement = sum(filtered_counts.values()) / sum(wlasl30_counts.values())
print(f"\nMejora en datos: {improvement:.1f}x")


COMPARACION DE DATASETS

Dataset                   Train      Val        Test       Total     
-----------------------------------------------------------------
WLASL30 original          242        56         34         332       
WLASL100 filtrado         242        56         34         332       

Mejora en datos: 1.0x


In [16]:
# ==========================================
# 11. GUARDAR METADATA DEL DATASET FILTRADO
# ==========================================

filtered_metadata = {
    'name': 'wlasl100_filtered30',
    'description': 'WLASL100 filtrado a las 30 clases de WLASL30',
    'source': 'wlasl100',
    'num_classes': 30,
    'glosses': list(TARGET_GLOSSES),
    'gloss_to_idx': gloss_to_idx,
    'idx_to_gloss': {str(v): k for k, v in gloss_to_idx.items()},
    'statistics': {
        'train': filtered_counts['train'],
        'val': filtered_counts['val'],
        'test': filtered_counts['test'],
        'total': sum(filtered_counts.values())
    }
}

with open(FILTERED_ROOT / 'metadata.json', 'w') as f:
    json.dump(filtered_metadata, f, indent=2)

print(f"Metadata guardado en {FILTERED_ROOT / 'metadata.json'}")

Metadata guardado en /home/fransando/Dev/tfg-sign-language-recognition/06_datasets/wlasl/landmarks/wlasl100_filtered30/metadata.json


---
# PARTE 3: ENTRENAR GRU CON DATOS ENRIQUECIDOS + LABEL SMOOTHING
---

In [17]:
# ==========================================
# 12. AUGMENTATION SUAVE
# ==========================================

class SkeletalAugmentationSuave:
    def __init__(self, rotation=10, scale=(0.9, 1.1), translation=0.1, noise=0.01, p=0.5):
        self.rotation = rotation
        self.scale = scale
        self.translation = translation
        self.noise = noise
        self.p = p
    
    def __call__(self, landmarks):
        landmarks = landmarks.copy()
        
        if random.random() < self.p:
            angle = np.radians(random.uniform(-self.rotation, self.rotation))
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            x, y = landmarks[:, :, 0].copy(), landmarks[:, :, 1].copy()
            landmarks[:, :, 0] = x * cos_a - y * sin_a
            landmarks[:, :, 1] = x * sin_a + y * cos_a
        
        if random.random() < self.p:
            scale = random.uniform(*self.scale)
            landmarks = landmarks * scale
        
        if random.random() < self.p:
            landmarks[:, :, 0] += random.uniform(-self.translation, self.translation)
            landmarks[:, :, 1] += random.uniform(-self.translation, self.translation)
        
        if random.random() < self.p:
            landmarks = landmarks + np.random.normal(0, self.noise, landmarks.shape)
        
        if random.random() < self.p * 0.5:
            landmarks[:, :, 0] = -landmarks[:, :, 0]
        
        return landmarks

print("Augmentation suave definida.")

Augmentation suave definida.


In [18]:
# ==========================================
# 13. CONFIGURACION PARA ENTRENAMIENTO FINAL
# ==========================================

CONFIG_FINAL = {
    'max_frames': 64,
    'input_size': 63,
    'num_classes': 30,
    
    # Arquitectura
    'model_type': 'GRU',
    'hidden_size': 256,
    'num_layers': 2,
    'dropout': 0.4,
    'bidirectional': True,
    
    # Training
    'batch_size': 32,
    'epochs': 150,
    'lr': 1e-3,
    'weight_decay': 5e-4,
    'label_smoothing': 0.1,  # NUEVO!
    
    'patience': 30,
}

print("Config final:")
print(f"  - Dataset: WLASL100 filtrado ({filtered_counts['train']} train)")
print(f"  - Modelo: {CONFIG_FINAL['model_type']}")
print(f"  - Label smoothing: {CONFIG_FINAL['label_smoothing']}")

Config final:
  - Dataset: WLASL100 filtrado (242 train)
  - Modelo: GRU
  - Label smoothing: 0.1


In [19]:
# ==========================================
# 14. CREAR LOADERS CON DATOS ENRIQUECIDOS
# ==========================================

train_aug = SkeletalAugmentationSuave()

train_dataset_enriched = PoseDataset(FILTERED_ROOT, 'train', gloss_to_idx,
                                     max_frames=64, augmentation=train_aug)
val_dataset_enriched = PoseDataset(FILTERED_ROOT, 'val', gloss_to_idx, max_frames=64)
test_dataset_enriched = PoseDataset(FILTERED_ROOT, 'test', gloss_to_idx, max_frames=64)

train_loader_enriched = DataLoader(train_dataset_enriched, batch_size=32, shuffle=True, num_workers=0)
val_loader_enriched = DataLoader(val_dataset_enriched, batch_size=32, shuffle=False)
test_loader_enriched = DataLoader(test_dataset_enriched, batch_size=32, shuffle=False)

print(f"Train enriquecido: {len(train_dataset_enriched)} samples")
print(f"Val enriquecido: {len(val_dataset_enriched)} samples")
print(f"Test enriquecido: {len(test_dataset_enriched)} samples")

Train enriquecido: 242 samples
Val enriquecido: 56 samples
Test enriquecido: 34 samples


In [20]:
# ==========================================
# 15. ENTRENAR MODELO FINAL
# ==========================================

# Crear modelo
model_final = PoseClassifier(CONFIG_FINAL).to(device)
print(f"Parametros: {sum(p.numel() for p in model_final.parameters()):,}")

# Loss con Label Smoothing!
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG_FINAL['label_smoothing'])
optimizer = AdamW(model_final.parameters(), lr=CONFIG_FINAL['lr'], 
                  weight_decay=CONFIG_FINAL['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG_FINAL['epochs'])

# Training loop
history = {'train_acc': [], 'val_top1': [], 'val_top5': []}
best_val = 0
patience_counter = 0

print("\n" + "=" * 60)
print("ENTRENAMIENTO: GRU + Datos Enriquecidos + Label Smoothing")
print("=" * 60)

for epoch in range(CONFIG_FINAL['epochs']):
    # Train
    model_final.train()
    correct, total = 0, 0
    for batch, labels in train_loader_enriched:
        batch, labels = batch.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_final(batch)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_final.parameters(), 1.0)
        optimizer.step()
        
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    scheduler.step()
    
    # Eval
    val_metrics, _, _ = evaluate_ensemble([model_final], val_loader_enriched)
    
    history['train_acc'].append(train_acc)
    history['val_top1'].append(val_metrics['top1_acc'])
    history['val_top5'].append(val_metrics['top5_acc'])
    
    # Save best
    if val_metrics['top1_acc'] > best_val:
        best_val = val_metrics['top1_acc']
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_final.state_dict(),
            'val_top1': best_val,
            'config': CONFIG_FINAL
        }, OUTPUT_DIR / 'best_model_enriched.pt')
    else:
        patience_counter += 1
    
    if patience_counter >= CONFIG_FINAL['patience']:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train: {train_acc:.1f}% | Val: {val_metrics['top1_acc']:.1f}% (Top-5: {val_metrics['top5_acc']:.1f}%) | Best: {best_val:.1f}%")

print(f"\nMejor Val Top-1: {best_val:.1f}% (epoch {best_epoch})")

Parametros: 2,128,158

ENTRENAMIENTO: GRU + Datos Enriquecidos + Label Smoothing
Epoch   1 | Train: 2.5% | Val: 7.1% (Top-5: 25.0%) | Best: 7.1%
Epoch  10 | Train: 20.7% | Val: 12.5% (Top-5: 41.1%) | Best: 12.5%
Epoch  20 | Train: 49.6% | Val: 17.9% (Top-5: 58.9%) | Best: 26.8%
Epoch  30 | Train: 80.2% | Val: 37.5% (Top-5: 67.9%) | Best: 37.5%
Epoch  40 | Train: 94.2% | Val: 32.1% (Top-5: 76.8%) | Best: 46.4%
Epoch  50 | Train: 99.2% | Val: 42.9% (Top-5: 78.6%) | Best: 48.2%
Epoch  60 | Train: 98.8% | Val: 51.8% (Top-5: 82.1%) | Best: 51.8%
Epoch  70 | Train: 99.2% | Val: 46.4% (Top-5: 75.0%) | Best: 51.8%
Epoch  80 | Train: 100.0% | Val: 51.8% (Top-5: 78.6%) | Best: 51.8%

Early stopping at epoch 90

Mejor Val Top-1: 51.8% (epoch 60)


In [21]:
# ==========================================
# 16. EVALUAR MODELO FINAL EN TEST
# ==========================================

# Cargar mejor modelo
ckpt = torch.load(OUTPUT_DIR / 'best_model_enriched.pt')
model_final.load_state_dict(ckpt['model_state_dict'])

# Evaluar en test original de WLASL30
test_metrics, _, _ = evaluate_ensemble([model_final], test_loader)

print("\n" + "=" * 50)
print("RESULTADOS EN TEST (WLASL30 original)")
print("=" * 50)
print(f"Top-1: {test_metrics['top1_acc']:.1f}%")
print(f"Top-5: {test_metrics['top5_acc']:.1f}%")


RESULTADOS EN TEST (WLASL30 original)
Top-1: 47.1%
Top-5: 79.4%


---
# PARTE 4: ENSEMBLE FINAL (3 MODELOS)
---

In [22]:
# ==========================================
# 17. ENSEMBLE DE 3 MODELOS
# ==========================================

print("\n" + "=" * 60)
print("ENSEMBLE FINAL: LSTM + GRU + GRU_Enriched")
print("=" * 60)

all_models = [lstm_model, gru_model, model_final]
model_names = ['LSTM', 'GRU', 'GRU_Enriched']

# Probar diferentes combinaciones de pesos
weight_configs_3 = [
    ([1/3, 1/3, 1/3], "Igual"),
    ([0.2, 0.3, 0.5], "Mas peso al nuevo"),
    ([0.2, 0.2, 0.6], "Mucho mas peso al nuevo"),
    ([0.3, 0.3, 0.4], "Ligeramente mas peso al nuevo"),
]

print("\nProbando diferentes pesos...")
best_ensemble = None
best_val_ensemble = 0

for weights, name in weight_configs_3:
    val_m, _, _ = evaluate_ensemble(all_models, val_loader, weights)
    test_m, _, _ = evaluate_ensemble(all_models, test_loader, weights)
    
    print(f"{name}: Val {val_m['top1_acc']:.1f}% / Test {test_m['top1_acc']:.1f}% (Top-5: {test_m['top5_acc']:.1f}%)")
    
    if val_m['top1_acc'] > best_val_ensemble:
        best_val_ensemble = val_m['top1_acc']
        best_ensemble = (weights, name, val_m, test_m)


ENSEMBLE FINAL: LSTM + GRU + GRU_Enriched

Probando diferentes pesos...
Igual: Val 50.0% / Test 58.8% (Top-5: 79.4%)
Mas peso al nuevo: Val 51.8% / Test 55.9% (Top-5: 79.4%)
Mucho mas peso al nuevo: Val 50.0% / Test 55.9% (Top-5: 79.4%)
Ligeramente mas peso al nuevo: Val 50.0% / Test 55.9% (Top-5: 79.4%)


In [23]:
# ==========================================
# 18. RESUMEN FINAL
# ==========================================

print("\n" + "=" * 70)
print("RESUMEN COMPARATIVO FINAL")
print("=" * 70)
print("")
print(f"{'Modelo':<35} {'Val Top-1':<12} {'Test Top-1':<12} {'Test Top-5':<12}")
print("-" * 70)

# Individuales
print(f"{'LSTM (original)':<35} {lstm_val['top1_acc']:.1f}%{'':<7} {lstm_test['top1_acc']:.1f}%{'':<7} {lstm_test['top5_acc']:.1f}%")
print(f"{'GRU suave':<35} {gru_val['top1_acc']:.1f}%{'':<7} {gru_test['top1_acc']:.1f}%{'':<7} {gru_test['top5_acc']:.1f}%")

# Nuevo modelo
new_val, _, _ = evaluate_ensemble([model_final], val_loader)
new_test, _, _ = evaluate_ensemble([model_final], test_loader)
print(f"{'GRU + Enriched + LabelSmooth':<35} {new_val['top1_acc']:.1f}%{'':<7} {new_test['top1_acc']:.1f}%{'':<7} {new_test['top5_acc']:.1f}%")

print("-" * 70)

# Ensembles
if best_config:
    print(f"{'Ensemble LSTM+GRU':<35} {best_config[2]['top1_acc']:.1f}%{'':<7} {best_config[3]['top1_acc']:.1f}%{'':<7} {best_config[3]['top5_acc']:.1f}%")

if best_ensemble:
    print(f"{'Ensemble 3 modelos':<35} {best_ensemble[2]['top1_acc']:.1f}%{'':<7} {best_ensemble[3]['top1_acc']:.1f}%{'':<7} {best_ensemble[3]['top5_acc']:.1f}%")

print("")
print("=" * 70)
print(f"OBJETIVO DEL TUTOR: Top-1 60-70%")
if best_ensemble and best_ensemble[3]['top1_acc'] >= 60:
    print("*** OBJETIVO ALCANZADO! ***")
else:
    best_test = max(lstm_test['top1_acc'], gru_test['top1_acc'], new_test['top1_acc'])
    if best_ensemble:
        best_test = max(best_test, best_ensemble[3]['top1_acc'])
    print(f"Mejor Test Top-1: {best_test:.1f}%")
    print(f"Faltan: {60 - best_test:.1f} puntos para el objetivo minimo")


RESUMEN COMPARATIVO FINAL

Modelo                              Val Top-1    Test Top-1   Test Top-5  
----------------------------------------------------------------------
LSTM (original)                     37.5%        41.2%        67.6%
GRU suave                           53.6%        38.2%        76.5%
GRU + Enriched + LabelSmooth        51.8%        47.1%        79.4%
----------------------------------------------------------------------
Ensemble LSTM+GRU                   50.0%        52.9%        79.4%
Ensemble 3 modelos                  51.8%        55.9%        79.4%

OBJETIVO DEL TUTOR: Top-1 60-70%
Mejor Test Top-1: 55.9%
Faltan: 4.1 puntos para el objetivo minimo


In [24]:
# ==========================================
# 19. GUARDAR RESULTADOS
# ==========================================

results = {
    'individual_models': {
        'lstm': {'val': lstm_val, 'test': lstm_test},
        'gru': {'val': gru_val, 'test': gru_test},
        'gru_enriched': {'val': new_val, 'test': new_test}
    },
    'ensemble_2': best_config[:2] if best_config else None,
    'ensemble_3': best_ensemble[:2] if best_ensemble else None,
    'history': history,
    'config': CONFIG_FINAL,
    'datasets': {
        'wlasl30': wlasl30_counts,
        'wlasl100_filtered': filtered_counts
    }
}

with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"Resultados guardados en: {OUTPUT_DIR}")

Resultados guardados en: /home/fransando/Dev/tfg-sign-language-recognition/05_experiments/ensemble_final


## Si todavia no llegas al 60%

Opciones adicionales:

1. **Attention mechanism**: Añadir atencion al GRU
2. **Mas epochs**: Subir a 200-300 epochs
3. **Mixup/Cutmix**: Data augmentation a nivel de batch
4. **Transformer**: Probar arquitectura de Transformer en vez de RNN